In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI
import os

In [3]:
# Инициализируем клиент OpenAI, но жестко перенаправляем его на официальный шлюз Google
client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.environ.get("GEMINI_API_KEY")
)



In [4]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [5]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [6]:
print("lesson pages:", len(documents))

lesson pages: 72


In [7]:
from minsearch import Index

index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)
index.fit(documents)

In [8]:
index.search('How does the agentic loop keep calling the model until it stops?', num_results=1)

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [9]:
# Переписываем класс под нашу структуру документов (filename и content)
class GeminiRAG:
    def __init__(self, index, llm_client):
        self.index = index
        self.llm_client = llm_client
        self.instructions = (
            "Your task is to answer questions from the course participants based on the provided context. "
            "Use the context to find relevant information and provide accurate answers. "
            "If the answer is not found in the context, respond with 'I don't know.'"
        )

    def search(self, query, num_results=5):
        # Адаптируем поиск: ищем по полю 'content' в индексе minsearch
        return self.index.search(
            query,
            filter_dict={}, # Убираем фильтр по курсу, так как в гитхабе его нет
            boost_dict={'content': 3.0}, # Даем максимальный вес тексту статьи
            num_results=num_results
        )

    def build_context(self, search_results):
        lines = []
        # Собираем контекст из новых полей: filename и content
        for doc in search_results:
            lines.append(f"File: {doc['filename']}")
            lines.append(f"Content: {doc['content']}")
            lines.append('')
        return '\n'.join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return f"QUESTION: {query}\n\nCONTEXT:\n{context}"

    def llm(self, prompt):
        response = self.llm_client.chat.completions.create(
            model="gemini-2.5-flash",
            messages=[
                {'role': 'system', 'content': self.instructions},
                {'role': 'user', 'content': prompt}
            ]
        )
        return response.choices[0].message.content

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        return self.llm(prompt)

# Инициализируем систему (передаём сюда объект индекса minsearch)
rag_assistant = GeminiRAG(index=index, llm_client=client)

# Запускаем финальный вопрос
query = "How does the agentic loop keep calling the model until it stops?"
print(rag_assistant.rag(query))


The agentic loop keeps calling the model until it stops by continuously checking if the model's response includes any function calls.

Here's how it works:
1.  **Initialize:** The loop starts with a set of `messages` including developer instructions and the user's question. An iteration counter is initialized.
2.  **Call the Model:** Inside a `while True` loop, the `openai_client.responses.create()` method is called with the current `messages` history and available `tools` (e.g., `search_tool`).
3.  **Process Response:** The model's response is received.
    *   **Append to History:** The model's output is appended to the `messages` history.
    *   **Check for Function Calls:** The code iterates through the items in the model's `response.output`.
        *   If an item is of `type == "function_call"`, it indicates the model wants to use a tool. The `make_call` helper function executes the specified tool (e.g., `search`) with the arguments provided by the model. The output of this tool

In [10]:
response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {'role': 'system', 'content': rag_assistant.instructions},
        {'role': 'user', 'content': rag_assistant.build_prompt(query, rag_assistant.search(query))}
    ]
)
print(response.usage.prompt_tokens)
print(response.usage.completion_tokens)

7952
408


In [11]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [12]:
print("chunks:", len(chunks))

chunks: 295


In [13]:
import minsearch

# Создаем отдельный индекс для чанков
chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

# Загружаем чанки вместо целых документов
chunk_index.fit(chunks)


In [14]:
# Направляем ассистента на новый chunk_index
rag_assistant_chunked = GeminiRAG(index=chunk_index, llm_client=client)

# Делаем тот же самый запрос
query = "How does the agentic loop keep calling the model until it stops?"

# Запрашиваем API и смотрим количество токенов
response_chunked = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {'role': 'system', 'content': rag_assistant_chunked.instructions},
        {'role': 'user', 'content': rag_assistant_chunked.build_prompt(query, rag_assistant_chunked.search(query))}
    ]
)

# Запоминаем новые токены
tokens_chunked = response_chunked.usage.prompt_tokens
print("Chunked_tokens:", tokens_chunked)


Chunked_tokens: 2605


In [15]:
import os
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [16]:
# Инициализируем глобальный счетчик перед запуском
search_counter = 0

def search(query: str) -> str:
    """
    Search the course knowledge base chunks for relevant documentation or lessons.
    Use this tool to find information about agentic loops and plain RAG.
    """
    global search_counter
    search_counter += 1  # Считаем каждый вызов инструмента агентом
    
    # Ищем по chunk_index
    results = chunk_index.search(
        query,
        filter_dict={},
        boost_dict={'content': 3.0},
        num_results=3
    )
    
    # Собираем текстовый ответ для агента
    lines = []
    for doc in results:
        lines.append(f"File: {doc['filename']}\nContent: {doc['content']}\n")
    
    return "\n".join(lines).strip()



In [17]:
# Регистрируем инструмент
agent_tools = Tools()
agent_tools.add_tool(search)

# Настраиваем интерактивный интерфейс
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

# Задаем системные инструкции (Промпт) из домашнего задания
instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
"""

gemini_client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.environ.get("GEMINI_API_KEY")
)

# Собираем раннер агента
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=gemini_client
)



In [18]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# 1. Сбрасываем глобальный счетчик перед запуском
search_counter = 0

# 2. Функция поиска (инструмент для агента)
def search(query: str) -> str:
    global search_counter
    search_counter += 1  # Считаем каждый вызов!
    print(f"   [Вызов поиска] Запрос: '{query}'")
    
    # Используем ваш chunk_index из шага Q4
    results = chunk_index.search(
        query,
        filter_dict={},
        boost_dict={'content': 3.0},
        num_results=3
    )
    
    lines = []
    for doc in results:
        lines.append(f"File: {doc['filename']}\nContent: {doc['content']}\n")
    return "\n".join(lines).strip()


# 3. Настройка клиента под бесплатный шлюз Gemini
client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.environ.get("GEMINI_API_KEY")
)

# 4. Описание инструмента в стандартном формате OpenAI JSON
tools_schema = [{
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the course knowledge base chunks for relevant documentation or lessons.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "The search keywords or query."}
            },
            "required": ["query"]
        }
    }
}]

# 5. Начальное состояние диалога (Инструкции из ДЗ + Вопрос)
messages = [
    {
        "role": "system", 
        "content": "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."
    },
    {
        "role": "user", 
        "content": "How does the agentic loop work, and how is it different from plain RAG?"
    }
]

# 6. Собственный агентный цикл (Agentic Loop)
print("Агент начал автономную работу...")
while True:
    response = client.chat.completions.create(
        model="gemini-2.5-flash",
        messages=messages,
        tools=tools_schema
    )
    
    assistant_message = response.choices[0].message
    messages.append(assistant_message)
    
    # Проверяем, решила ли модель вызвать инструмент поиска
    if assistant_message.tool_calls:
        for tool_call in assistant_message.tool_calls:
            if tool_call.function.name == "search":
                # Извлекаем сгенерированные агентом аргументы
                args = json.loads(tool_call.function.arguments)
                search_query = args.get("query")
                
                # Вызываем наш инструмент поиска
                tool_output = search(search_query)
                
                # Отправляем результаты поиска обратно модели
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": "search",
                    "content": tool_output
                })
    else:
        # Если модель больше не вызывает инструменты — она сформировала финальный ответ
        print("\n--- Финальный ответ Агента ---")
        print(assistant_message.content)
        break

print("\n" + "="*40)
print(f"Итоговое количество вызовов функции search: {search_counter}")
print("="*40)


Агент начал автономную работу...


   [Вызов поиска] Запрос: 'agentic loop'
   [Вызов поиска] Запрос: 'RAG vs agentic loop'

--- Финальный ответ Агента ---
The agentic loop is a core pattern in AI agents, characterized by a dynamic and iterative process. It involves a `while` loop where a Large Language Model (LLM) is repeatedly called. Within this loop, the LLM can decide to execute various tools based on its current understanding and the goal. The results from these tool calls are then fed back to the LLM, allowing it to refine its understanding, make further tool calls, or finally produce a conclusive answer. This loop continues until the LLM determines it has a final answer and no more tools are needed. Frameworks like Kestra's `AIAgent` or ToyAIKit abstract this manual `while` loop, handling conversation history, tool execution, and result management, enabling multi-turn conversations and adaptability to unforeseen conditions.

In contrast, plain Retrieval Augmented Generation (RAG) follows a more fixed sequence of

In [27]:
# Сбрасываем счетчик перед запуском теста
search_counter = 0

# Запускаем цикл агента с вопросом из задания
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

# Выводим итоговый ответ на вопрос домашнего задания
print("\n" + "="*40)
print(f"Count search: {search_counter}")
print("="*40)


AttributeError: 'OpenAI' object has no attribute 'model'